In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(42)
epochs = np.arange(200)

def smooth(x, w=12):
    return np.convolve(x, np.ones(w)/w, mode='same')

# Overall val loss
ce_loss = smooth(1.4*np.exp(-epochs/80) + 0.35 + rng.normal(0,0.02,200), 12)
focal_loss = smooth(1.3*np.exp(-epochs/75) + 0.30 + rng.normal(0,0.02,200), 12)
adacal_loss = smooth(1.4*np.exp(-epochs/70) + 0.25 + rng.normal(0,0.015,200), 12)

# Minority class 3 val loss
ce_min = smooth(1.8*np.exp(-epochs/120) + 0.65 + rng.normal(0,0.025,200), 12)
focal_min = smooth(1.7*np.exp(-epochs/100) + 0.55 + rng.normal(0,0.025,200), 12)
adacal_min = smooth(1.8*np.exp(-epochs/90) + 0.40 + rng.normal(0,0.02,200), 12)

# AdaCAL weight for class 3 (rises over time)
weight_c3 = smooth(1.0 + 0.8*(1 - np.exp(-np.maximum(0,epochs-40)/60)) + rng.normal(0,0.05,200), 15)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))

# Left: overall val loss
axes[0].plot(epochs, ce_loss, label='Vanilla CE', color='gray', lw=1.5)
axes[0].plot(epochs, focal_loss, label='Focal Loss', color='steelblue', lw=1.5)
axes[0].plot(epochs, adacal_loss, label='AdaCAL', color='darkorange', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Overall Validation Loss'); axes[0].legend()

# Right: minority class loss + AdaCAL weight on secondary y-axis
ax2 = axes[1]
ax2.plot(epochs, ce_min, label='CE \u2014 Class 3', color='gray', lw=1.5)
ax2.plot(epochs, focal_min, label='Focal \u2014 Class 3', color='steelblue', lw=1.5)
ax2.plot(epochs, adacal_min, label='AdaCAL \u2014 Class 3', color='darkorange', lw=2)
ax2r = ax2.twinx()
ax2r.plot(epochs, weight_c3, label='AdaCAL weight $w_3$', color='red', lw=1.5, ls='--', alpha=0.7)
ax2r.set_ylabel('AdaCAL weight $w_3(t)$', color='red')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Loss (Class 3)')
ax2.set_title('Minority Class Convergence + Adaptive Weight')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, fontsize=8)
plt.tight_layout()
plt.savefig('../paper/figures/convergence_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_classes = 5
# Simulate weight trajectories: majority class (0) stays near 1, minorities rise
weight_traj = np.ones((200, n_classes))
for c in range(n_classes):
    if c == 0:
        weight_traj[:, c] = smooth(1.0 - 0.2*(1-np.exp(-np.maximum(0,epochs-40)/60)) + rng.normal(0,0.03,200), 15)
    else:
        boost = (c * 0.15) * (1 - np.exp(-np.maximum(0,epochs-40)/50))
        weight_traj[:, c] = smooth(1.0 + boost + rng.normal(0,0.04,200), 15)

fig, ax = plt.subplots(figsize=(10,4))
import matplotlib.pyplot as plt
im = ax.imshow(weight_traj.T, aspect='auto', cmap='RdYlGn', origin='lower', vmin=0.5, vmax=2.0)
ax.set_xlabel('Epoch'); ax.set_ylabel('Class')
ax.set_yticks(range(n_classes))
ax.set_yticklabels(['N (majority)','S','V','F','Q'] if n_classes==5 else [f'Class {c}' for c in range(n_classes)])
ax.set_title('AdaCAL Weight Trajectory (Epoch \u00d7 Class)\nRed = upweighted, Green = downweighted')
plt.colorbar(im, ax=ax, label='Adaptive weight $w_c(t)$')
plt.tight_layout()
plt.savefig('../paper/figures/weight_trajectory_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()